# Getting Started: Local Diffing

This notebook is the shortest path from two DataFrames to a verdict you can inspect without writing files. It covers constructing a comparison, reading `DiffResult`, and isolating the rows behind a single column.

For YAML-driven CI, see the [CLI Quickstart](quickstart_cli.ipynb). For warehouse pushdown, see the [Configuration Guide](../configuration.md).

## 1. Two frames that look different and are not

A legacy export and a warehouse table of the same accounts. Currency symbols, a sentinel for missing status, and a one-cent rounding difference are the only things standing between them.

In [ ]:
import polars as pl

from veridelta import DiffConfig, DiffEngine, DiffRule

source = pl.DataFrame(
    {
        "legacy_id": [1, 2, 3],
        "status": ["Active", "N/A", "Closed"],
        "balance": ["$10.00", "$20.50", "$5.00"],
    }
)
target = pl.DataFrame(
    {
        "user_id": [1, 2, 3],
        "status": ["Active", None, "Closed"],
        "balance": [10.00, 20.48, 5.00],
    }
)
source, target

## 2. Declare the comparison

Rules are not applied in the order you write them. Every column follows the same nine-stage pipeline: sentinels, regex, whitespace and case, value map, padding, datetime, cast, then comparison. `cast_to` therefore always sees already-cleaned text.

In [ ]:
config = DiffConfig(
    primary_keys=["user_id"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(
            column_names=["balance"],
            regex_replace={"\\$": ""},
            cast_to="Float64",
            absolute_tolerance=0.05,
        ),
        DiffRule(
            column_names=["status"],
            null_values=["N/A"],
            treat_null_as_equal=True,
        ),
    ],
)

result = DiffEngine(config, source.lazy(), target.lazy()).run()
print(result.summary.report_summary)

## 3. Inspect the rows without writing artifacts

`run()` returns a `DiffResult`. `.summary` is the same value object as before; `.added`, `.removed`, and `.changed` are the frames the engine already materialized. `get_mismatches(column)` narrows the changed set to one column with both values side by side.

In [ ]:
print(result.compared_columns)
print(result.added)
print(result.removed)
print(result.changed)

# Empty here: both columns agreed after the rules ran.
result.get_mismatches("balance")

## 4. Isolate a real discrepancy

Drop the tolerance and the one-cent drift on row 2 surfaces. `get_mismatches` keeps only that column, so a row that drifted elsewhere does not appear.

In [ ]:
strict = DiffConfig(
    primary_keys=["user_id"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(column_names=["balance"], regex_replace={"\\$": ""}, cast_to="Float64"),
        DiffRule(column_names=["status"], null_values=["N/A"], treat_null_as_equal=True),
    ],
)
drifted = DiffEngine(strict, source.lazy(), target.lazy()).run()
print(drifted.summary.report_summary)
drifted.get_mismatches("balance")